# RQALL: Per-Well Accuracy (attn_recon models)

LaTeX-format **per-well** accuracy (majority vote of a well's pixel predictions, via `well_level_accuracy` -- same definition used throughout `RQ2_05_results.ipynb` / `RQ3_01_cv_results.ipynb` / `RQ3_02_loco_results.ipynb`) for the three evaluation protocols, restricted to the `cnn_gru_dual_attn_recon` model family only:

1. **RQ2** -- intra-chip 5-fold CV (like `RQ2_05`'s Ablation 6 section).
2. **RQ3 cross-validation** -- 5-fold CV pooling all 6 chips (like `RQ3_01`).
3. **RQ3 leave-one-chip-out (LOCO)** -- like `RQ3_02`, for the 10 attn_recon variants (5 training-time strategies x {no latent alignment, + latent alignment}). `RQ3_02`'s original 12-model `RQ3_2_COMPARISON` also includes the plain `cnn_gru_dual` baseline and its `_pc_recenter` counterpart; both are dropped here since they're not attn_recon variants.

Every loader function below is copied from its source notebook (see docstrings/comments for exactly which cell) and only trimmed to the attn_recon-only model list plus, in the LOCO section, extended to also compute well-level accuracy for the `_pc_recenter` variants (which `RQ3_02_loco_results.ipynb` does not currently do -- see `lofo_pc_recenter_accuracy_well`'s docstring).

## Setup

In [1]:
import os, sys, joblib, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              recall_score, confusion_matrix)

try:
    _nb = globals().get('__vsc_ipynb_file__')
    if _nb:
        os.chdir(os.path.dirname(os.path.dirname(_nb)))
except Exception:
    pass

%load_ext autoreload
%autoreload 2
import config
%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': '#e0e0e0',
    'grid.linestyle': '--',
    'grid.alpha': 0.7,
    'font.size': 10,
})
print("CWD:", os.getcwd())


[*] SUCCESS: TensorFlow is utilizing the GPU -> /physical_device:GPU:0

CWD: /vol/bitbucket/gk225/POC_DDM/gk_code/main


In [2]:
sys.path.insert(0, 'utils')
sys.path.insert(0, 'utils/model_training')
from model_utils import build_well_stratified_random_split, build_well_stratified_nfold_splits
from sklearn.preprocessing import LabelEncoder

def reconstruct_well_splits(y_full, well_ids_full, mask=None, n_splits=1, random_state=0):
    """Replays evaluate_outlier_filters' own mask -> rare-class-drop -> well-stratified-split
    procedure (utils/model_training/model_utils.py:863-916) exactly, so the returned test-set
    well_ids line up with the cached y_trues_/y_preds_AC_* arrays."""
    if mask is None:
        mask = np.ones(len(y_full), dtype=bool)
    y_m, w_m = y_full[mask], well_ids_full[mask]

    unique_classes, class_counts = np.unique(y_m, return_counts=True)
    rare = unique_classes[class_counts < 2]
    valid = ~np.isin(y_m, rare)
    y_v, w_v = y_m[valid], w_m[valid]

    if n_splits == 1:
        n_wells = len(np.unique(w_v))
        test_size = max(len(y_v) * 0.10, n_wells) / len(y_v)
        splits = list(build_well_stratified_random_split(
            y_v, w_v, test_size=test_size, random_state=random_state).values())
    else:
        actual_splits = min(n_splits, len(np.unique(w_v)))
        splits = list(build_well_stratified_nfold_splits(
            y_v, w_v, n_splits=actual_splits, random_state=random_state).values())

    return [(y_v[test_idx], w_v[test_idx]) for _, test_idx in splits]

def remap_well_splits(well_ids_full, y_full, mask, global_splits):
    """Mirrors _remap_global_splits (utils/model_training/model_utils.py:710) plus the
    rare-class-drop step inside evaluate_outlier_filters, for the cv_splits-supplied path
    used by 04_cross_dataset_training.py's kfold/lofo modes."""
    kept_global = np.where(mask)[0]
    y_m, w_m = y_full[mask], well_ids_full[mask]
    unique_classes, class_counts = np.unique(y_m, return_counts=True)
    rare = unique_classes[class_counts < 2]
    valid = ~np.isin(y_m, rare)
    kept_global = kept_global[valid]
    y_v, w_v = y_m[valid], w_m[valid]
    pos_lookup = {g: i for i, g in enumerate(kept_global)}

    def _remap_one(test_idx):
        local_test = np.array([pos_lookup[g] for g in test_idx if g in pos_lookup], dtype=int)
        return y_v[local_test], w_v[local_test]

    if isinstance(global_splits, dict):
        return {k: _remap_one(test_idx) for k, (_, test_idx) in global_splits.items()}
    return [_remap_one(test_idx) for _, test_idx in global_splits]

def well_level_accuracy(y_trues_flat, preds_flat, well_ids_test):
    """Majority-votes each well's pixel predictions into one verdict; accuracy over wells."""
    wells = np.unique(well_ids_test)
    n_true_mismatch = 0
    correct = 0
    for w in wells:
        m = well_ids_test == w
        true_vals = pd.Series(y_trues_flat[m])
        if true_vals.nunique() > 1:
            n_true_mismatch += 1
        true_label = true_vals.mode()[0]
        pred_label = pd.Series(preds_flat[m]).mode()[0]
        correct += int(true_label == pred_label)
    if n_true_mismatch:
        print(f"  [!] {n_true_mismatch} well(s) had a non-unanimous true label -- "
              f"split reconstruction likely mismatched training.")
    return correct / len(wells), len(wells)

def verified_well_ids(y_true_cached, folds, context=""):
    """folds: list of (y_values_fold, well_ids_fold) pairs. Returns the concatenated well_ids
    only if the concatenated y-values match y_true_cached exactly."""
    if not folds:
        return None
    y_check = np.concatenate([y for y, _ in folds])
    well_ids = np.concatenate([w for _, w in folds])
    if len(y_check) != len(y_true_cached) or not np.array_equal(y_check, y_true_cached):
        print(f"  [!] well-id reconstruction mismatch ({context}) -- skipping well_accuracy.")
        return None
    return well_ids

def verified_well_folds(cached_y_folds, folds, context=""):
    """Per-fold version of verified_well_ids -- for compute_fold_accuracy_stats."""
    if folds is None or len(folds) != len(cached_y_folds):
        print(f"  [!] well-id reconstruction mismatch ({context}: fold count) -- skipping well_accuracy.")
        return None
    out = []
    for (y_v, w_v), yt in zip(folds, cached_y_folds):
        if len(y_v) != len(yt) or not np.array_equal(y_v, yt):
            print(f"  [!] well-id reconstruction mismatch ({context}) -- skipping well_accuracy.")
            return None
        out.append(w_v)
    return out

In [3]:
# ── Paths ─────────────────────────────────────────────────────────────────────
EXP_FOLDER = config.FINAL_EXP_FOLDER
TARGET_FOLDERS = config.CROSS_DATASET_GROUPS['final_6_new']

def short_name(folder):
    m = re.search(r'DDM_0(\d)', folder)
    return f'Chip 0{m.group(1)}' if m else folder.split('_U_', 1)[1]

DATASET_NAMES = [short_name(f) for f in TARGET_FOLDERS]

# ── Labels ────────────────────────────────────────────────────────────────────
# Copied verbatim from RQ2_05_results.ipynb / RQ3_01_cv_results.ipynb / RQ3_02_loco_results.ipynb --
# the single source of truth for how a model key is displayed on any chart/table in this family
# of notebooks. This notebook only ever selects the attn_recon subset of these keys.
FAMILY_LABELS = {
    'cnn_gru_dual': 'CNN-BiGRU',
    'cnn_gru_dual_dann': 'CNN-BiGRU\n+ DANN',
    'cnn_gru_dual_supcon3': 'CNN-BiGRU\n+ Contrastive Loss (SC3)',
    'cnn_gru_dual_pc_recenter': 'CNN-BiGRU\n+ Latent Alignment',

    'cnn_gru_dual_attn_recon':   'CNN-BiGRU + Spatial Attn',
    'cnn_gru_dual_attn_recon_aug': 'CNN-BiGRU + Spatial Attn\n+ Temporal Aug',
    'cnn_gru_dual_attn_recon_dann': 'CNN-BiGRU + Spatial Attn\n+ DANN',
    'cnn_gru_dual_attn_recon_supcon3': 'CNN-BiGRU + Spatial Attn\n+ Contrastive Loss (SC3)',
    'cnn_gru_dual_attn_recon_pc_recenter': 'CNN-BiGRU + Spatial Attn\n+ Latent Alignment',

    'cnn_gru_dual_cosine_recon': 'CNN-BiGRU + Cosine Recon',
    'ccgd_arch_poc_st': 'CCGD-ST',

    'knn': 'kNN',
    'cnn_gru_dual_attn_recon_dann_pc_recenter': 'CNN-BiGRU + Spatial Attn\n+ DANN + Latent Alignment',
    'cnn_gru_dual_attn_recon_supcon3_pc_recenter': 'CNN-BiGRU + Spatial Attn\n+ Contrastive Loss (SC3) + Latent Alignment',
    'cnn_gru_dual_attn_recon_aug_pc_recenter': 'CNN-BiGRU + Spatial Attn\n+ Temporal Aug + Latent Alignment',
    'cnn_gru_dual_attn_recon_mtl': 'CNN-BiGRU + Spatial Attn\n+ MTL',
    'cnn_gru_dual_attn_recon_mtl_pc_recenter': 'CNN-BiGRU + Spatial Attn\n+ MTL + Latent Alignment',
    'cnn_gru_dual_attn_recon_coral': 'CNN-BiGRU + Spatial Attn\n+ CORAL',
    'cnn_gru_dual_attn_recon_coral_pc_recenter': 'CNN-BiGRU + Spatial Attn\n+ CORAL + Latent Alignment',
}

print(f"{len(TARGET_FOLDERS)} chips in group 'final_6_new':")
print("  " + "\n  ".join(DATASET_NAMES))

6 chips in group 'final_6_new':
  Chip 01
  Chip 02
  Chip 03
  Chip 04
  Chip 05
  Chip 06


In [4]:
def compute_metrics(perf, model_key, well_ids=None):
    """Return dict with accuracy/f1/auc/sensitivity/specificity, or None if missing.
    well_ids (optional): array aligned 1:1 with concatenate(perf['y_trues_']) -- adds
    well_accuracy/n_wells (majority-vote-per-well accuracy) alongside the pixel-level
    metrics above. Pass an already length/value-verified array (see verified_well_ids)."""
    pred_k = f'y_preds_AC_{model_key}_'
    prob_k = f'y_probs_AC_{model_key}_'
    if pred_k not in perf:
        return None

    y_true  = np.concatenate(perf['y_trues_'])
    y_pred  = np.concatenate(perf[pred_k])
    y_probs = np.concatenate(perf[prob_k]) if prob_k in perf else None
    n_cls   = len(np.unique(y_true))

    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    sens = recall_score(y_true, y_pred, average='macro', zero_division=0)

    if y_probs is not None and n_cls > 1:
        try:
            auc = roc_auc_score(y_true, y_probs, multi_class='ovr', average='macro')
        except Exception:
            auc = np.nan
    else:
        auc = np.nan

    cm = confusion_matrix(y_true, y_pred)
    specs = []
    for i in range(n_cls):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - cm[i, :].sum() - fp
        specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

    out = {'accuracy': acc, 'f1': f1, 'auc': auc,
           'sensitivity': sens, 'specificity': float(np.mean(specs))}

    if well_ids is not None:
        well_acc, n_wells = well_level_accuracy(y_true, y_pred, well_ids)
        out['well_accuracy'] = well_acc
        out['n_wells'] = n_wells

    return out


def macro_f1_sens_spec(y_true, y_pred):
    """F1/sensitivity (recall) macro-averaged, plus specificity (TN/(TN+FP) per class,
    macro-averaged)."""
    n_cls = len(np.unique(y_true))
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    sens = recall_score(y_true, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    specs = []
    for i in range(n_cls):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - cm[i, :].sum() - fp
        specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    return f1, sens, float(np.mean(specs))

## RQ2: Intra-chip per-well accuracy (attn_recon only)

Reads `ablations/ablation6_chip_outlier_model_ablation_performances.joblib` per chip (same source as `RQ2_05_results.ipynb`'s Ablation 6 section), restricted to `cnn_gru_dual_attn_recon` (baseline `cnn_gru_dual` is intentionally excluded).

In [5]:
# ABL6_MODELS restricted to the attn_recon variant only (per this notebook's brief --
# 'cnn_gru_dual' baseline is intentionally excluded here; see RQ2_05_results.ipynb for
# the full baseline-vs-attn_recon comparison).
ABL6_MODELS = ['cnn_gru_dual_attn_recon']
ABL6_CURVE_TYPE = 'ori_curve_sg_p4_norm'
ABL6_FILTERS = [None]  # f_sngl_g13.sh's ablation6 run only trains the baseline (no filter)


def compute_fold_accuracy_stats(perf, model_key, well_ids_folds=None):
    """Mean/std accuracy ACROSS folds (not pooled) -- matches ablation6_chip_outlier_model_ablation.py's
    own log line (utils/model_training/model_utils.py:2476-2486). well_ids_folds (optional):
    list of well_ids arrays, one per fold -- adds well_accuracy_mean/std."""
    pred_k = f'y_preds_AC_{model_key}_'
    if pred_k not in perf:
        return None
    fold_accs = [accuracy_score(yt, yp) for yt, yp in zip(perf['y_trues_'], perf[pred_k])]
    out = {'accuracy_mean': np.mean(fold_accs) * 100, 'accuracy_std': np.std(fold_accs) * 100}

    fold_f1s, fold_senss, fold_specs = [], [], []
    for yt, yp in zip(perf['y_trues_'], perf[pred_k]):
        f1, sens, spec = macro_f1_sens_spec(yt, yp)
        fold_f1s.append(f1); fold_senss.append(sens); fold_specs.append(spec)
    out['f1_mean'] = np.mean(fold_f1s) * 100;             out['f1_std'] = np.std(fold_f1s) * 100
    out['sensitivity_mean'] = np.mean(fold_senss) * 100;  out['sensitivity_std'] = np.std(fold_senss) * 100
    out['specificity_mean'] = np.mean(fold_specs) * 100;  out['specificity_std'] = np.std(fold_specs) * 100

    if well_ids_folds is not None:
        fold_well_accs = []
        for yt, yp, w in zip(perf['y_trues_'], perf[pred_k], well_ids_folds):
            wa, _ = well_level_accuracy(yt, yp, w)
            fold_well_accs.append(wa)
        out['well_accuracy_mean'] = np.mean(fold_well_accs) * 100
        out['well_accuracy_std'] = np.std(fold_well_accs) * 100
    return out


_ABL6_ARRAYS_CACHE = {}

def _abl6_training_arrays(exp_path):
    """y_full/well_ids_full/features_df exactly as ablation6_chip_outlier_model_ablation.py's
    run_one() derives them for ABL6_CURVE_TYPE, cached per exp_path."""
    if exp_path in _ABL6_ARRAYS_CACHE:
        return _ABL6_ARRAYS_CACHE[exp_path]
    training_data = joblib.load(exp_path / config.TRAINING_DATA_PATH)
    dataset_name = list(training_data["dataset_name"])
    kinetic_features = training_data["kinetic_features"]
    Y_well = training_data["Y_well"]

    keep = [i for i, n in enumerate(dataset_name)
            if not n.startswith("avg_") and not n.startswith("original_fitted_stretched")]
    dataset_name = [dataset_name[i] for i in keep]
    kinetic_features = [kinetic_features[i] for i in keep]

    metadata_df = pd.DataFrame(training_data["metadata"])
    well_ids_full = (metadata_df["well_id"].values if "well_id" in metadata_df.columns
                      else np.array(Y_well).copy())

    label_mappings = config.get_label_mappings(exp_path)
    if exp_path.name in label_mappings:
        mapping = label_mappings[exp_path.name]
        Y_well = [mapping.get(w, w) for w in Y_well]
    encoder = LabelEncoder()
    y_full = encoder.fit_transform(Y_well)

    target_name = config.CURVE_TYPE_ALIASES.get(ABL6_CURVE_TYPE, ABL6_CURVE_TYPE)
    result = ((y_full, well_ids_full, kinetic_features[dataset_name.index(target_name)], encoder.classes_)
              if target_name in dataset_name else (None, None, None, None))
    _ABL6_ARRAYS_CACHE[exp_path] = result
    return result


def abl6_well_folds(exp_path, filt):
    y_full, well_ids_full, features_df, _ = _abl6_training_arrays(exp_path)
    if y_full is None:
        return None
    if filt is None:
        mask = np.ones(len(y_full), dtype=bool)
    elif filt not in features_df.columns:
        return None
    else:
        mask = (features_df[filt] == 1).fillna(False).values
    return reconstruct_well_splits(y_full, well_ids_full, mask=mask, n_splits=5)


def load_ablation6_all():
    rows = []
    for folder in TARGET_FOLDERS:
        dataset = short_name(folder)
        exp_path = Path(EXP_FOLDER) / folder
        path = exp_path / 'ablations' / 'ablation6_chip_outlier_model_ablation_performances.joblib'
        if not path.exists():
            print(f'[WARN] missing {path}')
            continue
        data = joblib.load(path)
        for filt in ABL6_FILTERS:
            perf = data.get(filt)
            if perf is None:
                continue
            well_folds = abl6_well_folds(exp_path, filt)
            for model in ABL6_MODELS:
                m = compute_metrics(perf, model)
                if m is None:
                    continue
                saved_well_ids_folds = perf.get('well_ids_test_')
                if saved_well_ids_folds is not None:
                    well_ids_folds = saved_well_ids_folds
                elif well_folds is not None:
                    well_ids_folds = verified_well_folds(perf['y_trues_'], well_folds,
                                                         context=f'{folder}/{filt}/{model}')
                else:
                    well_ids_folds = None
                stats = compute_fold_accuracy_stats(perf, model, well_ids_folds=well_ids_folds)
                rows.append(dict(dataset=dataset, outlier_filter=filt, model=model, **stats, **m))
    return pd.DataFrame(rows)

abl6_df = load_ablation6_all()
print(f"Loaded {len(abl6_df)} rows  |  "
      f"{abl6_df.dataset.nunique()} datasets  "
      f"{abl6_df.model.nunique()} model(s)")
abl6_df[['dataset', 'model', 'accuracy_mean', 'accuracy_std', 'well_accuracy_mean', 'well_accuracy_std']]

Loaded 6 rows  |  6 datasets  1 model(s)


,dataset,model,accuracy_mean,accuracy_std,well_accuracy_mean,well_accuracy_std
0,Chip 01,cnn_gru_dual_attn_recon,99.951173,0.083235,100.0,0.0
1,Chip 02,cnn_gru_dual_attn_recon,99.717505,0.268560,100.0,0.0
2,Chip 03,cnn_gru_dual_attn_recon,99.814970,0.179448,100.0,0.0
3,Chip 04,cnn_gru_dual_attn_recon,98.816748,0.502213,100.0,0.0
4,Chip 05,cnn_gru_dual_attn_recon,99.902017,0.116135,100.0,0.0
5,Chip 06,cnn_gru_dual_attn_recon,99.505012,0.482130,100.0,0.0


In [6]:
def build_per_well_single_model_latex_table(df, model, dataset_order, caption, label,
                                             value_col='well_accuracy_mean', std_col='well_accuracy_std'):
    """Single-model, per-dataset PER-WELL accuracy table: one row per dataset, plus a
    Macro Avg row (mean +/- std across datasets). Used identically by the RQ2 (intra-chip)
    and RQ3 cross-validation sections below -- only the input df/caption/label differ."""
    sub = df[df.model == model]
    lines = [
        '\\begin{table}[htbp]',
        '    \\centering',
        f'    \\caption{{{caption}}}',
        f'    \\label{{{label}}}',
        '    \\small',
        '    \\begin{tabular}{@{}lr@{}}',
        '    \\toprule',
        '    \\textbf{Dataset} & \\textbf{Well Accuracy} \\\\',
        '    \\midrule',
    ]
    vals = []
    for ds in dataset_order:
        r = sub[sub.dataset == ds] if 'dataset' in sub.columns else sub[sub.fold == ds]
        if r.empty or pd.isna(r[value_col].iloc[0]):
            lines.append(f'    {ds} & -- \\\\')
            continue
        v_mean, v_std = r[value_col].iloc[0], r[std_col].iloc[0]
        vals.append(v_mean)
        lines.append(f'    {ds} & {v_mean:.2f}\\% $\\pm$ {v_std:.2f}\\% \\\\')
    lines.append('    \\midrule')
    if vals:
        macro_mean, macro_std = np.mean(vals), np.std(vals)
        lines.append(f'    \\textbf{{Macro Avg}} & \\textbf{{{macro_mean:.2f}\\% $\\pm$ {macro_std:.2f}\\%}} \\\\')
    else:
        lines.append('    \\textbf{Macro Avg} & -- \\\\')
    lines += ['    \\bottomrule', '    \\end{tabular}', '\\end{table}']
    return '\n'.join(lines)


rq2_per_well_latex = build_per_well_single_model_latex_table(
    abl6_df, model='cnn_gru_dual_attn_recon', dataset_order=DATASET_NAMES,
    caption=('Intra-chip, per-well accuracy for CNN-BiGRU + Spatial Attn across five-fold '
             'cross-validation. Each chip is trained and evaluated independently; a well\'s '
             'predicted label is its pixels\' majority vote. Results are reported as mean '
             '$\\pm$ standard deviation across folds, and all values are percentages.'),
    label='tab:rq2_attn_recon_well_accuracy',
)
print(rq2_per_well_latex)

\begin{table}[htbp]
    \centering
    \caption{Intra-chip, per-well accuracy for CNN-BiGRU + Spatial Attn across five-fold cross-validation. Each chip is trained and evaluated independently; a well's predicted label is its pixels' majority vote. Results are reported as mean $\pm$ standard deviation across folds, and all values are percentages.}
    \label{tab:rq2_attn_recon_well_accuracy}
    \small
    \begin{tabular}{@{}lr@{}}
    \toprule
    \textbf{Dataset} & \textbf{Well Accuracy} \\
    \midrule
    Chip 01 & 100.00\% $\pm$ 0.00\% \\
    Chip 02 & 100.00\% $\pm$ 0.00\% \\
    Chip 03 & 100.00\% $\pm$ 0.00\% \\
    Chip 04 & 100.00\% $\pm$ 0.00\% \\
    Chip 05 & 100.00\% $\pm$ 0.00\% \\
    Chip 06 & 100.00\% $\pm$ 0.00\% \\
    \midrule
    \textbf{Macro Avg} & \textbf{100.00\% $\pm$ 0.00\%} \\
    \bottomrule
    \end{tabular}
\end{table}


## RQ3 cross-validation: per-well accuracy (attn_recon only)

5-fold CV pooling all 6 `final_6_new` chips together (same source as `RQ3_01_cv_results.ipynb`), restricted to `cnn_gru_dual_attn_recon` (`knn` and `cnn_gru_dual` are excluded).

In [7]:
import importlib

# Reuse 06b's own results-file discovery instead of reimplementing it, so this section
# always matches what 06b_cross_dataset_prediction_report.py actually reads.
b06 = importlib.import_module("06b_cross_dataset_prediction_report")
p04 = importlib.import_module("04_cross_dataset_training")

CV_MODE_STR    = "kfold5"
CV_FILTER      = "noamp_remove"
CV_CURVE_TYPES = ["ori_curve_sg_p4_norm"]
CV_CURVE_SHORT = {"ori_curve_sg_p4_norm": "sgp4_norm"}
CV_TRAIN_CENTER_FRAC = 0.5  # matches --train_center_frac on f_k5_g13.sh

# attn_recon only, per this notebook's brief (RQ3_01_cv_results.ipynb compares
# 'knn'/'cnn_gru_dual'/'cnn_gru_dual_attn_recon' -- only the last is kept here).
CV_MODELS = ['cnn_gru_dual_attn_recon']

def cv_group_dir(group_name):
    return Path(EXP_FOLDER) / "cross_dataset_cv" / group_name

def load_cv_results(group_name, curve_type, train_center_frac=CV_TRAIN_CENTER_FRAC):
    group_dir = cv_group_dir(group_name)
    legacy_path = b06.find_results_path(group_dir, CV_MODE_STR, curve_type)
    return b06.load_partitioned(group_dir, CV_MODE_STR, curve_type, legacy_path=legacy_path,
                                train_center_frac=train_center_frac)


def _cv_well_folds(group_name, curve_type):
    """Test-set well_ids for every fold_x of this group/curve_type's kfold5 split, mirroring
    04_cross_dataset_training.py's kfold path exactly."""
    exp_paths = [Path(EXP_FOLDER, name) for name in config.CROSS_DATASET_GROUPS[group_name]]
    combined = p04.combine_group(exp_paths, group_name, curve_type=curve_type)
    if combined is None:
        print(f"  [!] {group_name}/{curve_type}: combine_group() returned nothing -- well_accuracy skipped.")
        return None
    if combined["well_ids"] is None:
        print(f"  [!] {group_name}/{curve_type}: no pixel_row_idx/pixel_col_idx metadata (well_ids unavailable) -- well_accuracy skipped.")
        return None
    y_full = LabelEncoder().fit_transform(combined["Y_mapped"])
    cv_splits = p04.build_nfold_splits(y_full, well_ids=combined["well_ids"], n_splits=5)
    mask = (p04.is_amplifying_mask(combined["curves"]) if CV_FILTER == p04.NOAMP_FILTER_NAME
            else np.ones(len(y_full), dtype=bool))
    return remap_well_splits(combined["well_ids"], y_full, mask, cv_splits)


def load_cv_accuracy(group_indices=None, train_center_frac=CV_TRAIN_CENTER_FRAC):
    if group_indices is not None:
        groups = [list(config.CROSS_DATASET_GROUPS)[i] for i in group_indices]
    else:
        groups = list(config.CROSS_DATASET_GROUPS)

    rows = []
    for group_name in groups:
        for curve_type in CV_CURVE_TYPES:
            lofo_results = load_cv_results(group_name, curve_type, train_center_frac=train_center_frac)
            if not lofo_results:
                print(f"[WARN] No results for group {group_name} / curve {curve_type}")
                continue
            fold_labels = sorted([k for k in lofo_results if k.startswith("fold_")],
                                  key=lambda s: int(s.split("_")[1]))

            _needs_legacy = any(
                lofo_results[fold].get(CV_FILTER, {}).get('well_ids_test_') is None
                for fold in fold_labels if lofo_results[fold].get(CV_FILTER) is not None
            )
            well_folds = None
            if _needs_legacy:
                try:
                    well_folds = _cv_well_folds(group_name, curve_type)
                except Exception as e:
                    print(f"  [!] {group_name}/{curve_type}: could not reconstruct well_ids ({e}) -- well_accuracy skipped.")
                    well_folds = None

            for fold in fold_labels:
                res_entry = lofo_results[fold].get(CV_FILTER)
                if res_entry is None:
                    continue
                y_true_full = np.concatenate(res_entry["y_trues_"])
                saved_well_ids = res_entry.get('well_ids_test_')
                if saved_well_ids is not None:
                    well_ids_test = np.concatenate(saved_well_ids)
                else:
                    well_ids_test = (verified_well_ids(y_true_full, [well_folds[fold]],
                                                       context=f'{group_name}/{curve_type}/{fold}')
                                     if well_folds is not None and fold in well_folds else None)
                for model in CV_MODELS:
                    preds_key = config.MODEL_KEY_MAP.get(model, (None,))[0]
                    if preds_key is None or preds_key not in res_entry:
                        continue
                    y_pred = np.concatenate(res_entry[preds_key])
                    row = dict(group=group_name, curve_type=CV_CURVE_SHORT.get(curve_type, curve_type),
                              fold=fold, model=model,
                              accuracy=accuracy_score(y_true_full, y_pred) * 100,
                              n_test=len(y_true_full), well_accuracy=np.nan, n_wells=np.nan)
                    if well_ids_test is not None:
                        well_acc, n_wells = well_level_accuracy(y_true_full, y_pred, well_ids_test)
                        row["well_accuracy"] = well_acc * 100
                        row["n_wells"] = n_wells
                    rows.append(row)
    return pd.DataFrame(rows, columns=['group', 'curve_type', 'fold', 'model', 'accuracy', 'n_test', 'well_accuracy', 'n_wells'])

cv_df = load_cv_accuracy(group_indices=[list(config.CROSS_DATASET_GROUPS).index('final_6_new')])
print(f"Loaded {len(cv_df)} rows  |  {cv_df.fold.nunique()} folds  |  {cv_df.model.nunique()} model(s)")
cv_df

Loaded 5 rows  |  5 folds  |  1 model(s)


,group,curve_type,fold,model,accuracy,n_test,well_accuracy,n_wells
0,final_6_new,sgp4_norm,fold_0,cnn_gru_dual_attn_recon,95.921302,17383,100.0,47
1,final_6_new,sgp4_norm,fold_1,cnn_gru_dual_attn_recon,96.680997,17445,100.0,47
2,final_6_new,sgp4_norm,fold_2,cnn_gru_dual_attn_recon,95.461335,17406,100.0,47
3,final_6_new,sgp4_norm,fold_3,cnn_gru_dual_attn_recon,96.185004,17405,100.0,47
4,final_6_new,sgp4_norm,fold_4,cnn_gru_dual_attn_recon,95.948276,17400,100.0,47


In [8]:
def build_cv_per_well_latex_table(cv_df, group_name, curve_type, model, model_labels, caption, label):
    """5-fold CV pools all chips together (unlike RQ2's per-chip intra-chip CV or RQ3's
    per-held-out-chip LOFO), so there is one well-accuracy number here: mean +/- std of
    well_accuracy across the 5 folds, for the single (attn_recon) model this notebook covers."""
    sub = cv_df[(cv_df.group == group_name) & (cv_df.curve_type == curve_type) & (cv_df.model == model)]
    mean_v, std_v = sub['well_accuracy'].mean(), sub['well_accuracy'].std()
    row_label = model_labels.get(model, model).replace(chr(10), ' ')

    lines = [
        '\\begin{table}[htbp]',
        '    \\centering',
        f'    \\caption{{{caption}}}',
        f'    \\label{{{label}}}',
        '    \\small',
        '    \\begin{tabular}{@{}lr@{}}',
        '    \\toprule',
        '    \\textbf{Model} & \\textbf{Well Accuracy} \\\\',
        '    \\midrule',
    ]
    if pd.isna(mean_v):
        lines.append(f'    {row_label} & -- \\\\')
    else:
        lines.append(f'    {row_label} & {mean_v:.2f}\\% $\\pm$ {std_v:.2f}\\% \\\\')
    lines += ['    \\bottomrule', '    \\end{tabular}', '\\end{table}']
    return '\n'.join(lines)


rq3cv_per_well_latex = build_cv_per_well_latex_table(
    cv_df, group_name='final_6_new', curve_type='sgp4_norm', model='cnn_gru_dual_attn_recon',
    model_labels=FAMILY_LABELS,
    caption=('5-fold cross-chip validation, per-well accuracy for CNN-BiGRU + Spatial Attn '
             'on final\\_6\\_new (SG p=4, normalized). A well\'s predicted label is its pixels\' '
             'majority vote. Reported as mean $\\pm$ standard deviation across the 5 folds.'),
    label='tab:rq3_cv_attn_recon_well_accuracy',
)
print(rq3cv_per_well_latex)

\begin{table}[htbp]
    \centering
    \caption{5-fold cross-chip validation, per-well accuracy for CNN-BiGRU + Spatial Attn on final\_6\_new (SG p=4, normalized). A well's predicted label is its pixels' majority vote. Reported as mean $\pm$ standard deviation across the 5 folds.}
    \label{tab:rq3_cv_attn_recon_well_accuracy}
    \small
    \begin{tabular}{@{}lr@{}}
    \toprule
    \textbf{Model} & \textbf{Well Accuracy} \\
    \midrule
    CNN-BiGRU + Spatial Attn & 100.00\% $\pm$ 0.00\% \\
    \bottomrule
    \end{tabular}
\end{table}


## RQ3 leave-one-chip-out (LOCO): per-well accuracy (attn_recon only)

Same source as `RQ3_02_loco_results.ipynb`. `RQ3_2_BASE_MODELS`/`PC_RECENTER_BASES` are trimmed to the 5 attn_recon variants (dropping the plain `cnn_gru_dual` baseline and its `_pc_recenter` counterpart from the original 12-model comparison).

In [9]:
p08   = importlib.import_module("08_cross_dataset_predict_new_chip")
vis07 = importlib.import_module("07_attribution_vis_all")
rio   = importlib.import_module("cross_dataset_result_io")

import tensorflow as tf
tf.config.optimizer.set_jit(False)

LOFO_MODE_STR    = "lofo"
LOFO_CURVE_ALIGN = "pc_ttp"
LOFO_PC_TTP_ANCHOR = "min"
LOFO_FRAC        = 0.5  # matches --train_center_frac on f_lf_g13.sh
LOFO_FILTER      = "noamp_remove"
LOFO_CURVE_TYPES = ["ori_curve_sg_p4_norm"]
LOFO_CURVE_SHORT = {"ori_curve_sg_p4_norm": "sgp4_norm"}
LOFO_GROUP_NAMES = ["final_6_new"]

# attn_recon only, per this notebook's brief -- 'cnn_gru_dual' (and its _pc_recenter
# counterpart) are dropped from RQ3_02_loco_results.ipynb's original 12-model
# RQ3_2_COMPARISON, leaving the 10 attn_recon variants below (5 training-time
# strategies x {no latent alignment, + latent alignment}).
RQ3_2_BASE_MODELS = [
    "cnn_gru_dual_attn_recon",
    "cnn_gru_dual_attn_recon_dann", "cnn_gru_dual_attn_recon_supcon3",
    "cnn_gru_dual_attn_recon_aug", "cnn_gru_dual_attn_recon_mtl",
]
PC_RECENTER_BASES = [
    "cnn_gru_dual_attn_recon", "cnn_gru_dual_attn_recon_dann",
    "cnn_gru_dual_attn_recon_supcon3", "cnn_gru_dual_attn_recon_aug",
    "cnn_gru_dual_attn_recon_mtl",
]
RQ3_2_COMPARISON = RQ3_2_BASE_MODELS + [f"{m}_pc_recenter" for m in PC_RECENTER_BASES]

def lofo_group_dir(group_name):
    return b06.alignment_dir(cv_group_dir(group_name), LOFO_CURVE_ALIGN, LOFO_PC_TTP_ANCHOR)

def load_lofo_results(group_name, curve_type, train_center_frac=LOFO_FRAC):
    out_dir = lofo_group_dir(group_name)
    legacy_path = b06.find_results_path(out_dir, LOFO_MODE_STR, curve_type)
    return b06.load_partitioned(out_dir, LOFO_MODE_STR, curve_type, legacy_path=legacy_path,
                                train_center_frac=train_center_frac)

print(f"{len(RQ3_2_COMPARISON)} attn_recon model variants defined "
      f"({len(RQ3_2_BASE_MODELS)} base + {len(PC_RECENTER_BASES)} + latent alignment).")

10 attn_recon model variants defined (5 base + 5 + latent alignment).

In [10]:
# Separate cache file from RQ3_02_loco_results.ipynb's -- this notebook's cached rows
# carry extra well-accuracy fields that older cache entries won't have (see the
# self-healing "well_acc_pc_recenter" in cached["row"] check in lofo_pc_recenter_accuracy_well
# below), so the two notebooks' caches are kept independent to avoid cross-invalidation.
PC_RECENTER_CACHE_NAME = "pc_recenter_sweep_cache_well.joblib"

def _pc_cache_path(group_name):
    return Path(EXP_FOLDER) / "cross_dataset_cv" / group_name / PC_RECENTER_CACHE_NAME

def load_pc_cache(group_name):
    path = _pc_cache_path(group_name)
    if not path.exists():
        return {}
    try:
        return joblib.load(path)
    except Exception:
        return {}

def save_pc_cache(group_name, cache):
    path = _pc_cache_path(group_name)
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(cache, path, compress=3)

def _mtime(path):
    try:
        return path.stat().st_mtime
    except FileNotFoundError:
        return None

def pc_cache_signature(model_path, out_dir, curve_type, held_out_chip=None):
    align_dir = config.cross_dataset_alignment_dir(out_dir, held_out_chip)
    return (_mtime(model_path),
           _mtime(align_dir / config.CROSS_DATASET_RESAMPLER_PATH.format(curve_type=curve_type)),
           _mtime(align_dir / config.CROSS_DATASET_PC_TTP_RECIPE_PATH.format(curve_type=curve_type)))

In [11]:
def lofo_ground_truth(chip_name, Y_well_raw, class_names):
    mapping = config.LABEL_MAPPINGS[chip_name]
    y_true = np.array([mapping.get(w, w) for w in Y_well_raw])
    if class_names:
        cn = list(class_names)
        y_true = np.array([next((c for c in cn if y == c or y.startswith(c + '-') or c.startswith(y + '-')), y)
                           for y in y_true])
    return y_true


def lofo_pc_recenter_accuracy_well(group_name, curve_type, base_model, chip_path, exp_paths_all, class_names, cache,
                                   train_center_frac=LOFO_FRAC):
    """Extends RQ3_02_loco_results.ipynb's lofo_pc_recenter_accuracy with per-well accuracy
    for both the baseline and +latent-alignment predictions. well_ids comes straight out of
    align_new_chip's own align_result -- same row order as curves/Y_well_raw/the predict_new_chip
    output, so well_ids[valid]/y_true[valid]/pred[valid] are already aligned 1:1; no split
    reconstruction needed here (predict_new_chip returns one prediction per input row, it doesn't
    drop or reorder anything -- see 08_cross_dataset_predict_new_chip.py). NOTE: like the
    original lofo_pc_recenter_accuracy, this does NOT apply noamp_remove filtering to the
    accuracy/well_accuracy figures -- it stays consistent with acc_pc_recenter's own existing
    pixel set (only PC wells are excluded via `valid`), rather than introducing a second,
    differently-filtered pixel population for the same model."""
    out_dir = lofo_group_dir(group_name)
    chip_name = chip_path.name
    fold_label = f"lofo_{chip_name}"
    model_dir = out_dir / "model_interpretation" / fold_label
    model_path = model_dir / f"{base_model}_{LOFO_FILTER}_{curve_type}{rio.frac_suffix(train_center_frac)}_model.keras"
    if not model_path.exists():
        return None

    cache_key = (curve_type, LOFO_CURVE_ALIGN, LOFO_PC_TTP_ANCHOR, train_center_frac, LOFO_FILTER, base_model, chip_name)
    sig = pc_cache_signature(model_path, out_dir, curve_type, held_out_chip=chip_name)
    cached = cache.get(cache_key)
    if cached is not None and cached["sig"] == sig and "well_acc_pc_recenter" in cached["row"]:
        return cached["row"]

    align_result = p08.align_new_chip(chip_path, out_dir, curve_type, LOFO_CURVE_ALIGN, LOFO_PC_TTP_ANCHOR,
                                      group_name=group_name, held_out_chip=chip_name)
    if align_result is None:
        return None
    curves, resampler, Y_well_raw, pc_curves_aligned, coords, well_ids = align_result

    loaded = vis07.load_saved_models(model_dir, LOFO_FILTER, len(resampler.t_grid), curve_type=curve_type,
                                     model_names=[base_model], train_center_frac=train_center_frac)
    if base_model not in loaded:
        return None
    model = loaded[base_model]
    if p08._is_spatial(base_model) and (coords is None or well_ids is None):
        return None

    y_true = lofo_ground_truth(chip_name, Y_well_raw, class_names)
    valid = y_true != "PC"
    exp_paths_train = [p for p in exp_paths_all if p.name != chip_name]

    probs_base, _ = p08.predict_new_chip(model, base_model, curves, coords, well_ids, pc_curves_aligned,
                                         exp_paths_train, out_dir, curve_type, LOFO_FILTER, LOFO_CURVE_ALIGN,
                                         pc_recenter=False, held_out_chip=chip_name)
    pred_base = np.array(class_names)[np.argmax(probs_base, axis=1)]
    acc_base = (pred_base[valid] == y_true[valid]).mean() if valid.any() else float('nan')
    cm_base = confusion_matrix(y_true[valid], pred_base[valid], labels=class_names) if valid.any() else None
    if valid.any() and well_ids is not None:
        well_acc_base, n_wells_base = well_level_accuracy(y_true[valid], pred_base[valid], well_ids[valid])
    else:
        well_acc_base, n_wells_base = float('nan'), 0

    acc_recenter = float('nan')
    cm_recenter = None
    well_acc_recenter, n_wells_recenter = float('nan'), 0
    try:
        probs_r, _ = p08.predict_new_chip(model, base_model, curves, coords, well_ids, pc_curves_aligned,
                                          exp_paths_train, out_dir, curve_type, LOFO_FILTER, LOFO_CURVE_ALIGN,
                                          pc_recenter=True, force_rerun=True, held_out_chip=chip_name)
        pred_r = np.array(class_names)[np.argmax(probs_r, axis=1)]
        acc_recenter = (pred_r[valid] == y_true[valid]).mean() if valid.any() else float('nan')
        cm_recenter = confusion_matrix(y_true[valid], pred_r[valid], labels=class_names) if valid.any() else None
        if valid.any() and well_ids is not None:
            well_acc_recenter, n_wells_recenter = well_level_accuracy(y_true[valid], pred_r[valid], well_ids[valid])
    except ValueError:
        pass   # no PC snapshot for this chip -- leave acc_recenter/well_acc_recenter as NaN

    row = {"acc_baseline": acc_base * 100, "acc_pc_recenter": acc_recenter * 100, "n_pixels": int(valid.sum()),
          "well_acc_baseline": well_acc_base * 100 if not np.isnan(well_acc_base) else np.nan,
          "well_acc_pc_recenter": well_acc_recenter * 100 if not np.isnan(well_acc_recenter) else np.nan,
          "n_wells": n_wells_recenter or n_wells_base,
          "cm_base": cm_base, "cm_recenter": cm_recenter, "class_names": list(class_names)}
    cache[cache_key] = {"sig": sig, "row": row}
    save_pc_cache(group_name, cache)

    del model, loaded
    tf.keras.backend.clear_session()
    return row


def _lofo_well_ids_for_chip(group_name, curve_type, chip_path, class_names):
    """noamp_remove-filtered, label-verified well_ids for one held-out chip's LOFO test set --
    used for the base (non-recenter) models' well_accuracy, matching what evaluate_outlier_filters
    actually filtered res_entry down to."""
    out_dir = lofo_group_dir(group_name)
    chip_name = chip_path.name
    align_result = p08.align_new_chip(chip_path, out_dir, curve_type, LOFO_CURVE_ALIGN, LOFO_PC_TTP_ANCHOR,
                                      group_name=group_name, held_out_chip=chip_name)
    if align_result is None:
        return None
    curves, resampler, Y_well_raw, pc_curves_aligned, coords, well_ids = align_result
    if well_ids is None:
        print(f"  [!] {group_name}/{curve_type}/{chip_name}: no pixel_row_idx/pixel_col_idx metadata (well_ids unavailable) -- well_accuracy skipped.")
        return None

    keep = p08.cdt.is_amplifying_mask(curves)
    Y_well_raw, well_ids = Y_well_raw[keep], well_ids[keep]

    y_true_str = lofo_ground_truth(chip_name, Y_well_raw, class_names)
    label_to_idx = {c: i for i, c in enumerate(class_names)}
    valid = np.array([lbl in label_to_idx for lbl in y_true_str])
    y_true_int = np.array([label_to_idx.get(lbl, -1) for lbl in y_true_str])
    return y_true_int[valid], well_ids[valid]

In [12]:
def load_lofo_accuracy_well(groups=None, group_indices=None, train_center_frac=LOFO_FRAC):
    if groups is None:
        if group_indices is not None:
            groups = [list(config.CROSS_DATASET_GROUPS)[i] for i in group_indices]
        else:
            groups = list(config.CROSS_DATASET_GROUPS)
    rows = []
    for group_name in groups:
        exp_paths_all = [Path(EXP_FOLDER, name) for name in config.CROSS_DATASET_GROUPS[group_name]]
        pc_cache = None

        for curve_type in LOFO_CURVE_TYPES:
            lofo_results = load_lofo_results(group_name, curve_type, train_center_frac=train_center_frac)
            if not any(k != "full_data" for k in lofo_results):
                continue
            if pc_cache is None:
                pc_cache = load_pc_cache(group_name)

            for chip_path in exp_paths_all:
                fold_entry = lofo_results.get(f"lofo_{chip_path.name}", {})
                res_entry = fold_entry.get(LOFO_FILTER)
                class_names = fold_entry.get("class_names")
                fold = short_name(chip_path.name)

                if res_entry is not None:
                    y_true_full = np.concatenate(res_entry["y_trues_"])
                    saved_well_ids = res_entry.get('well_ids_test_')
                    if saved_well_ids is not None:
                        well_ids_test = np.concatenate(saved_well_ids)
                    else:
                        well_ids_test = None
                        if class_names is not None:
                            try:
                                well_check = _lofo_well_ids_for_chip(group_name, curve_type, chip_path, class_names)
                            except Exception as e:
                                print(f"  [!] {group_name}/{curve_type}/{fold}: could not reconstruct well_ids ({e}) -- well_accuracy skipped.")
                                well_check = None
                            if well_check is not None:
                                well_ids_test = verified_well_ids(y_true_full, [well_check],
                                                                  context=f'{group_name}/{curve_type}/{fold}')

                    for model in RQ3_2_BASE_MODELS:
                        preds_key = config.MODEL_KEY_MAP.get(model, (None,))[0]
                        if preds_key is None or preds_key not in res_entry:
                            continue
                        y_pred = np.concatenate(res_entry[preds_key])
                        row = dict(group=group_name, curve_type=LOFO_CURVE_SHORT.get(curve_type, curve_type),
                                  fold=fold, model=model,
                                  accuracy=accuracy_score(y_true_full, y_pred) * 100,
                                  n_test=len(y_true_full), well_accuracy=np.nan, n_wells=np.nan)
                        if well_ids_test is not None:
                            well_acc, n_wells = well_level_accuracy(y_true_full, y_pred, well_ids_test)
                            row["well_accuracy"] = well_acc * 100
                            row["n_wells"] = n_wells
                        rows.append(row)

                if class_names is None:
                    continue
                for base_model in PC_RECENTER_BASES:
                    result = lofo_pc_recenter_accuracy_well(group_name, curve_type, base_model, chip_path,
                                                            exp_paths_all, class_names, pc_cache,
                                                            train_center_frac=train_center_frac)
                    if result is None:
                        continue
                    rows.append(dict(group=group_name, curve_type=LOFO_CURVE_SHORT.get(curve_type, curve_type),
                                     fold=fold, model=f"{base_model}_pc_recenter",
                                     accuracy=result["acc_pc_recenter"],
                                     n_test=result["n_pixels"],
                                     well_accuracy=result["well_acc_pc_recenter"], n_wells=result["n_wells"]))
    return pd.DataFrame(rows, columns=['group', 'curve_type', 'fold', 'model', 'accuracy', 'n_test', 'well_accuracy', 'n_wells'])


lofo_df = load_lofo_accuracy_well(group_indices=[list(config.CROSS_DATASET_GROUPS).index('final_6_new')])
print(f"Loaded {len(lofo_df)} rows  |  "
      f"{lofo_df.fold.nunique()} held-out chips  |  {lofo_df.model.nunique()} model variants "
      f"(expected {len(RQ3_2_COMPARISON)})")
lofo_df

  [*] LOFO_EXCLUDE_WELL_MAPPING['final_6_new']: dropping 1686 samples from wells [8, 9] (y_label={8: 'PC', 9: 'NC-ALL'}, y_concentration={8: '0e+00', 9: '0e+00'}) for D20260827_E00_C00_F4500KHz_U_DDM_01_final_final


  [PC-TTP align] new chip TTP=321.26  anchor=281.24  shift=40.02


/vol/bitbucket/gk225/venv_poc_ddm/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 70 variables whereas the saved optimizer has 66 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/vol/bitbucket/gk225/venv_poc_ddm/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 66 variables whereas the saved optimizer has 0 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


  [*] Saved PC reference embedding -> /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final/cross_dataset_cv/final_6_new/curve_alignment_pc_ttp/anchor_min/model_interpretation/lofo_D20260827_E00_C00_F4500KHz_U_DDM_01_final_final/pc_reference_embedding_cnn_gru_dual_attn_recon_noamp_remove_ori_curve_sg_p4_norm.joblib


  [*] LOFO_EXCLUDE_WELL_MAPPING['final_6_new']: dropping 1686 samples from wells [8, 9] (y_label={8: 'PC', 9: 'NC-ALL'}, y_concentration={8: '0e+00', 9: '0e+00'}) for D20260827_E00_C00_F4500KHz_U_DDM_01_final_final


...

  [*] LOFO_EXCLUDE_WELL_MAPPING['final_6_new']: dropping 1842 samples from wells [8, 9] (y_label={8: 'PC', 9: 'NC-ALL'}, y_concentration={8: '0e+00', 9: '0e+00'}) for D20260827_E00_C00_F4500KHz_U_DDM_03_final_final


  [PC-TTP align] new chip TTP=281.24  anchor=290.28  shift=0.00


  [*] Saved PC reference embedding -> /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final/cross_dataset_cv/final_6_new/curve_alignment_pc_ttp/anchor_min/model_interpretation/lofo_D20260827_E00_C00_F4500KHz_U_DDM_03_final_final/pc_reference_embedding_cnn_gru_dual_attn_recon_dann_noamp_remove_ori_curve_sg_p4_norm.joblib


  [*] LOFO_EXCLUDE_WELL_MAPPING['final_6_new']: dropping 1842 samples from wells [8, 9] (y_label={8: 'PC', 9: 'NC-ALL'}, y_concentration={8: '0e+00', 9: '0e+00'}) for D20260827_E00_C00_F4500KHz_U_DDM_03_final_final


  [PC-TTP align] new chip TTP=281.24  anchor=290.28  shift=0.00


### LaTeX table -- LOCO per-well accuracy, Baseline vs. training-time strategies vs. + Latent Space Alignment

Single combined table (matches the target format exactly): bold marks the best result in each held-out-chip column (and in Macro Avg), computed across all 10 listed conditions together, not per-section.

In [ ]:
_LOCO_BASE_LABEL = 'CNN-BiGRU + Spatial Attn'
_LOCO_STRATEGY_LABEL = {
    'cnn_gru_dual_attn_recon_dann': '+ DANN',
    'cnn_gru_dual_attn_recon_supcon3': '+ SupCon',
    'cnn_gru_dual_attn_recon_aug': '+ Temporal Aug',
    'cnn_gru_dual_attn_recon_mtl': '+ MTL',
}


def _loco_well_matrix(lofo_df, models, pc_recenter, dataset_order):
    """One row per model: per-chip well_accuracy (single value -- one LOFO fold per chip,
    no averaging needed there) plus a Macro Avg/Std across the 6 held-out chips."""
    rows = []
    for m in models:
        key = f"{m}_pc_recenter" if pc_recenter else m
        sub = lofo_df[(lofo_df.model == key) & (lofo_df.curve_type == "sgp4_norm")
                      & (lofo_df.group == "final_6_new")]
        per_chip = [sub.loc[sub.fold == ds, "well_accuracy"].iloc[0]
                   if (sub.fold == ds).any() else np.nan for ds in dataset_order]
        vals = np.array(per_chip, dtype=float)
        row = {"ModelKey": m, "PcRecenter": pc_recenter}
        row.update(dict(zip(dataset_order, per_chip)))
        row["Macro Avg (%)"] = np.nanmean(vals) if not np.all(np.isnan(vals)) else np.nan
        row["Macro Std (%)"] = np.nanstd(vals) if not np.all(np.isnan(vals)) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def build_loco_combined_well_latex_table(lofo_df, base_models, pc_recenter_bases, dataset_order, caption, label):
    base_df = _loco_well_matrix(lofo_df, base_models, pc_recenter=False, dataset_order=dataset_order)
    la_df   = _loco_well_matrix(lofo_df, pc_recenter_bases, pc_recenter=True, dataset_order=dataset_order)
    all_df  = pd.concat([base_df, la_df], ignore_index=True)

    best_col = {}
    for ds in dataset_order:
        vals = all_df[ds].astype(float)
        best_col[ds] = int(vals.idxmax()) if vals.notna().any() else None
    macro_vals = all_df['Macro Avg (%)'].astype(float)
    best_macro = int(macro_vals.idxmax()) if macro_vals.notna().any() else None

    def fmt_cell(row_idx, row, ds):
        v = row[ds]
        if pd.isna(v):
            return '--'
        s = f'{v:.2f}'
        return f'\\textbf{{{s}}}' if best_col[ds] == row_idx else s

    def fmt_macro(row_idx, row):
        if pd.isna(row['Macro Avg (%)']):
            return '--'
        s = f"{row['Macro Avg (%)']:.2f} $\\pm$ {row['Macro Std (%)']:.2f}"
        return f'\\textbf{{{s}}}' if best_macro == row_idx else s

    def row_label(model_key, is_section_start):
        if is_section_start:
            return _LOCO_BASE_LABEL
        return f'\\quad {_LOCO_STRATEGY_LABEL.get(model_key, model_key)}'

    lines = [
        '\\begin{table}[htbp]',
        '    \\centering',
        f'    \\caption{{{caption}}}',
        f'    \\label{{{label}}}',
        '    \\footnotesize',
        '    \\setlength{\\tabcolsep}{1.5pt}',
        '    \\begin{tabular}{@{}l ' + 'r' * len(dataset_order) + ' r@{}}',
        '    \\toprule',
        '    & \\multicolumn{' + str(len(dataset_order)) + '}{c}{\\textbf{Held-Out Chip}} & \\\\',
        f'    \\cmidrule(lr){{2-{1 + len(dataset_order)}}}',
        '    \\textbf{Model} & ' + ' & '.join(f'\\textbf{{{ds}}}' for ds in dataset_order)
        + ' & \\textbf{Macro Avg} \\\\',
        '    \\midrule',
        f'    \\multicolumn{{{len(dataset_order) + 2}}}{{l}}{{\\textbf{{Baseline}}}} \\\\',
    ]
    for i, m in enumerate(base_models):
        row_idx = i  # position within all_df / base_df (base rows come first)
        row = all_df.loc[row_idx]
        cells = [fmt_cell(row_idx, row, ds) for ds in dataset_order]
        lbl = row_label(m, is_section_start=(i == 0))
        lines.append(f'    {lbl} & {" & ".join(cells)} & {fmt_macro(row_idx, row)} \\\\')
        if i == 0:
            lines.append('    \\addlinespace')
            lines.append(f'    \\multicolumn{{{len(dataset_order) + 2}}}{{l}}{{\\textbf{{Training time strategy}}}} \\\\')
    lines.append('    \\addlinespace')
    lines.append(f'    \\multicolumn{{{len(dataset_order) + 2}}}{{l}}{{\\textbf{{+ Latent Space Alignment}}}} \\\\')
    for i, m in enumerate(pc_recenter_bases):
        row_idx = len(base_models) + i  # la_df rows follow base_df rows in all_df
        row = all_df.loc[row_idx]
        cells = [fmt_cell(row_idx, row, ds) for ds in dataset_order]
        lbl = row_label(m, is_section_start=(i == 0))
        lines.append(f'    {lbl} & {" & ".join(cells)} & {fmt_macro(row_idx, row)} \\\\')
    lines += ['    \\bottomrule', '    \\end{tabular}', '\\end{table}']
    return '\n'.join(lines)


rq3loco_per_well_latex = build_loco_combined_well_latex_table(
    lofo_df, RQ3_2_BASE_MODELS, PC_RECENTER_BASES, DATASET_NAMES,
    caption=('Leave one chip out per-well accuracy for each held-out chip, for the CNN-BiGRU '
             '+ Spatial Attn baseline, each training time strategy, and the same conditions '
             'with the inference time control anchored latent space alignment additionally '
             'applied. A well\'s predicted label is its pixels\' majority vote. The Macro Avg '
             'column reports the mean $\\pm$ standard deviation across the six held-out chips, '
             'and all values are percentages. Bold marks the best result in each held-out chip '
             'across all listed conditions.'),
    label='tab:lofo_attn_recon_well_accuracy',
)
print(rq3loco_per_well_latex)